The project will be a 2-D coupled spring system. Let us start with a simpler case of having two masses on a spring that
are coupled with eachother in the x-direction. But those springs are also suspended by springs in the y-axis, both one spring above and one spring below. 


To start, let us create a class in python called spring_parameters, which will define the spring constant k_i, and damping coefficient b_i,
for each spring. We will have a seperate class called masses, which defines the masses m_i that the springs are attached to.

In [ ]:
import numpy as np

WALLS = ("left", "right", "top", "bottom")


class box:
    """Rectangular box [0, W] x [0, H] whose sides are the walls."""

    def __init__(self, W, H):
        self.W, self.H = float(W), float(H)

    def anchor(self, wall, r):
        """Point where a spring from a mass resting at r = (x, y) attaches to `wall`.
        Left/right attach at the mass's height; top/bottom directly above/below it."""
        x, y = r
        return np.array({"left": (0, y), "right": (self.W, y),
                         "top": (x, self.H), "bottom": (x, 0)}[wall], dtype=float)


class spring_parameters:
    """Spring constants k_i, damping coefficients b_i, and the two ends of each spring.

    Each end is either a mass index (int) or a wall name from WALLS.
    """

    def __init__(self, k, b, ends):
        self.k = np.asarray(k, dtype=float)
        self.b = np.asarray(b, dtype=float)
        self.ends = list(ends)
        if not (self.k.size == self.b.size == len(self.ends)):
            raise ValueError(f"got {self.k.size} k's, {self.b.size} b's, {len(self.ends)} ends")
        for a, c in self.ends:
            if a in WALLS and c in WALLS:
                raise ValueError(f"spring ({a}, {c}) connects two walls")
            for e in (a, c):
                if not (isinstance(e, int) or e in WALLS):
                    raise ValueError(f"bad spring end {e!r}")
        self.n = self.k.size

    def attach(self, box, r0):
        """Fix the wall anchor points and set rest lengths L0 from the resting positions r0,
        so every spring is unstretched when the masses sit at r0."""
        r0 = np.asarray(r0, dtype=float)
        self.anchor = {}                       # spring index -> fixed wall point
        for s, (a, c) in enumerate(self.ends):
            if a in WALLS:
                self.anchor[s] = box.anchor(a, r0[c])
            if c in WALLS:
                self.anchor[s] = box.anchor(c, r0[a])
        P, Q = self.endpoints(r0)
        self.L0 = np.linalg.norm(Q - P, axis=1)

    def endpoints(self, r):
        """Positions (n_springs, 2) of both ends of every spring, for mass positions r."""
        pos = lambda s, e: r[e] if isinstance(e, int) else self.anchor[s]
        P = np.array([pos(s, a) for s, (a, c) in enumerate(self.ends)])
        Q = np.array([pos(s, c) for s, (a, c) in enumerate(self.ends)])
        return P, Q

    def forces(self, r, v):
        """Net spring + damping force on each mass, shape (n_masses, 2).
        Spring s pulls its ends together with k (L - L0) along its length, and its
        dashpot adds b * (rate of stretching). Wall ends are fixed (zero velocity)."""
        end = lambda s, e: (r[e], v[e]) if isinstance(e, int) else (self.anchor[s], np.zeros(2))
        F = np.zeros_like(r)
        for s, (a, c) in enumerate(self.ends):
            (pa, va), (pc, vc) = end(s, a), end(s, c)
            d = pc - pa
            L = np.linalg.norm(d)
            u = d / L                                   # unit vector from end a to end c
            f = (self.k[s] * (L - self.L0[s]) + self.b[s] * np.dot(vc - va, u)) * u
            if isinstance(a, int): F[a] += f            # stretched spring pulls a toward c
            if isinstance(c, int): F[c] -= f            # ... and c toward a
        return F

    def energy(self, r):
        """Total elastic energy  sum_s  k_s (L_s - L0_s)^2 / 2."""
        P, Q = self.endpoints(r)
        L = np.linalg.norm(Q - P, axis=1)
        return 0.5 * np.sum(self.k * (L - self.L0) ** 2)


class masses:
    """Masses m_i that the springs attach to."""

    def __init__(self, m):
        self.m = np.asarray(m, dtype=float)
        if np.any(self.m <= 0):
            raise ValueError("masses must be positive")
        self.n = self.m.size


In [ ]:
# Trial 1: two masses in a square W x H box (W = H, so the layout is symmetric).  Mass 0 is on the left, mass 1 on the right.
#   left wall -- m0 -- m1 -- right wall,  plus each mass hangs between the top and bottom walls.
W, H = 3.0, 3.0
container = box(W, H)
r0 = np.array([[W / 3, H / 2],                  # resting positions: thirds across, mid-height
               [2 * W / 3, H / 2]])

ends = [
    (0, 1),                                     # coupling spring between the masses
    ("left", 0), (0, "top"), (0, "bottom"),     # mass 0 to its walls
    (1, "right"), (1, "top"), (1, "bottom"),    # mass 1 to its walls
]
springs = spring_parameters(k=np.ones(7), b=np.zeros(7), ends=ends)
springs.attach(container, r0)
m = masses([1.0, 1.0])

# Checks: 7 springs, 4 per mass, and rest lengths match the geometry
# (horizontal springs span W/3, vertical springs span H/2).
assert springs.n == 7
for i in range(m.n):
    assert sum(i in e for e in springs.ends) == 4, f"mass {i} is not attached to 4 springs"
assert np.allclose(springs.L0, [W/3, W/3, H/2, H/2, W/3, H/2, H/2])
print("L0 =", springs.L0)


Now I want to visualize the system in 2-D. I will use matplotlib to create a simple animation of the two masses and their springs. The animation will show the motion of the masses as they oscillate due to the forces from the springs and damping.

In [ ]:
import matplotlib.pyplot as plt


def zigzag(p, q, coils=8, width=0.08):
    """Zigzag line from p to q, for drawing a spring."""
    t = np.linspace(0, 1, 2 * coils + 3)
    side = np.r_[0, (-1) ** np.arange(2 * coils + 1), 0]   # straight leads at both ends
    d = q - p
    nrm = np.array([-d[1], d[0]]) / np.linalg.norm(d)
    return p + np.outer(t, d) + width * np.outer(side, nrm)


def draw_system(ax, r):
    """Draw the box, every spring, and the masses at positions r. Returns the artists."""
    ax.add_patch(plt.Rectangle((0, 0), container.W, container.H, fill=False, lw=2))
    P, Q = springs.endpoints(r)
    lines = [ax.plot(*zigzag(p, q).T, color="gray", lw=1)[0] for p, q in zip(P, Q)]
    dots = ax.scatter(*r.T, s=300 * m.m, color="tab:blue", zorder=3)
    ax.set(xlim=(-0.2, container.W + 0.2), ylim=(-0.2, container.H + 0.2), aspect="equal")
    return lines, dots


# Check: with every spring at its rest length, no mass feels a force.
assert np.allclose(springs.forces(r0, np.zeros_like(r0)), 0)

fig, ax = plt.subplots(figsize=(4, 4))
draw_system(ax, r0)
for i, (x, y) in enumerate(r0):
    ax.annotate(f"$m_{i}$", (x, y), xytext=(0, 14), textcoords="offset points", ha="center")
ax.set_title("At rest, no applied force")
plt.show()


Now apply a force to one of the masses and animate the motion. To start, drive mass 0 along x with $F(t) = F_0\cos(\omega_d t)$ and no damping. Units: $k = m = 1$, so frequencies are in units of $\sqrt{k/m}$.

### Equations of motion

Each mass $i$ has position $\mathbf r_i = (x_i, y_i)$. A spring $s$ joins mass $i$ to an end $j$, which is either the other mass or a fixed wall anchor $\mathbf p_s$ (with $\dot{\mathbf p}_s = 0$). Write

$$\mathbf d_{s} = \mathbf r_i - \mathbf r_j, \qquad L_s = |\mathbf d_{s}|, \qquad \hat{\mathbf u}_{s} = \mathbf d_{s} / L_s .$$

Spring $s$ pushes on mass $i$ along its own length. Its stretch $L_s - L_{0,s}$ gives a Hooke's-law force, and its dashpot resists the rate of stretching $\dot L_s = (\dot{\mathbf r}_i - \dot{\mathbf r}_j)\cdot\hat{\mathbf u}_s$:

$$m_i\,\ddot{\mathbf r}_i = \sum_{s\,\ni\, i} \Big[-k_s\,(L_s - L_{0,s}) \;-\; b_s\,(\dot{\mathbf r}_i - \dot{\mathbf r}_j)\cdot\hat{\mathbf u}_s\Big]\,\hat{\mathbf u}_s \;+\; \mathbf F_i(t).$$

The sum runs over the 4 springs attached to mass $i$. Every $L_{0,s}$ equals that spring's length when the masses sit at rest at $\mathbf r_i^{0}$, so all forces vanish there. These equations are nonlinear because $L_s$ and $\hat{\mathbf u}_s$ depend on the positions, and the code integrates them in full.

**Small oscillations.** For small displacements $\xi_i = x_i - x_i^0$ and $\eta_i = y_i - y_i^0$, only springs lying along a direction resist motion in that direction. To first order, the $x$ and $y$ motions separate. Label the springs $c$ (coupling), $L, R$ (side walls), and $T_i, B_i$ (top and bottom of mass $i$):

$$\begin{aligned}
m_0\,\ddot\xi_0 &= -k_L\,\xi_0 - k_c\,(\xi_0 - \xi_1) - b_L\,\dot\xi_0 - b_c\,(\dot\xi_0 - \dot\xi_1) + F_{0x}(t), \\
m_1\,\ddot\xi_1 &= -k_R\,\xi_1 - k_c\,(\xi_1 - \xi_0) - b_R\,\dot\xi_1 - b_c\,(\dot\xi_1 - \dot\xi_0) + F_{1x}(t), \\
m_0\,\ddot\eta_0 &= -(k_{T_0} + k_{B_0})\,\eta_0 - (b_{T_0} + b_{B_0})\,\dot\eta_0 + F_{0y}(t), \\
m_1\,\ddot\eta_1 &= -(k_{T_1} + k_{B_1})\,\eta_1 - (b_{T_1} + b_{B_1})\,\dot\eta_1 + F_{1y}(t).
\end{aligned}$$

The coupling spring links the two masses only in $x$. It starts at its rest length, so to first order it gives no restoring force when the masses move in $y$. With all $k = m = 1$ and $b = 0$, the natural frequencies are $\omega = 1$ (both masses moving together in $x$), $\omega = \sqrt3$ (masses moving opposite in $x$), and $\omega = \sqrt2$ (each mass in $y$).


In [ ]:
from scipy.integrate import solve_ivp
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

driven, F0, w_drive = 0, 0.2, 0.7          # which mass, force amplitude, drive frequency
T, n_frames = 30.0, 200                    # simulated time and number of animation frames


def F_applied(t):
    """External force on each mass: F0 cos(w t) in x on the driven mass only."""
    F = np.zeros((m.n, 2))
    F[driven, 0] = F0 * np.cos(w_drive * t)
    return F


def rhs(t, y):
    """State y = [positions, velocities, work done by F_applied]."""
    r, v = y[:2 * m.n].reshape(m.n, 2), y[2 * m.n:4 * m.n].reshape(m.n, 2)
    Fa = F_applied(t)
    a = (springs.forces(r, v) + Fa) / m.m[:, None]
    return np.r_[v.ravel(), a.ravel(), np.sum(Fa * v)]


y0 = np.r_[r0.ravel(), np.zeros(2 * m.n), 0.0]
t = np.linspace(0, T, n_frames)
sol = solve_ivp(rhs, (0, T), y0, t_eval=t, method="DOP853", rtol=1e-10, atol=1e-12)
r_t = sol.y[:2 * m.n].T.reshape(-1, m.n, 2)
v_t = sol.y[2 * m.n:4 * m.n].T.reshape(-1, m.n, 2)
work = sol.y[-1]

# Check (work-energy theorem): with no damping, the change in kinetic + spring energy
# must equal the work done by the applied force.
E = np.array([0.5 * np.sum(m.m[:, None] * v ** 2) + springs.energy(r) for r, v in zip(r_t, v_t)])
print(f"max |E - E(0) - W| = {np.max(np.abs(E - E[0] - work)):.2e}   (max E = {E.max():.3f})")
assert np.allclose(E - E[0], work, atol=1e-8)

# Animation: the system on the left, x (solid) and y (dashed) displacements of every mass on the right.
def animate(t, r_t, title):
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(9, 4), dpi=60)
    lines, dots = draw_system(ax, r_t[0])
    ax.set_title(title)
    disp = r_t - r0
    traces = [ax2.plot([], [], color=f"C{i}", ls=ls, label=f"${c}_{i}$")[0]
              for i in range(m.n) for c, ls in [("x", "-"), ("y", "--")]]
    lim = 1.1 * np.abs(disp).max()
    ax2.set(xlim=(t[0], t[-1]), ylim=(-lim, lim), xlabel="t", ylabel="displacement from rest")
    ax2.legend(loc="upper right", ncol=2)
    fig.tight_layout()

    def update(j):
        P, Q = springs.endpoints(r_t[j])
        for line, p, q in zip(lines, P, Q):
            line.set_data(*zigzag(p, q).T)
        dots.set_offsets(r_t[j])
        for k, tr in enumerate(traces):
            tr.set_data(t[:j + 1], disp[:j + 1, k // 2, k % 2])
        return lines + [dots] + traces

    anim = FuncAnimation(fig, update, frames=len(t), interval=50, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())


animate(t, r_t, f"mass {driven} driven in x,  $\\omega_d$ = {w_drive}")


Let us now sweep the driving frequency $\omega_d$ from 0 to 3 and plot the amplitude of the driven mass $0$ in $x$. We will use a driving force $F(t) = F_0\cos(\omega_d t)$ with $F_0 = 0.05$ and a small damping $b = 0.05$ on every spring (with $F_0 = 1$ the masses would hit the walls near resonance). The amplitude is defined as the maximum displacement of mass $0$ in $x$ after transients have died out.

In [ ]:
F0_sweep, b_sweep = 0.05, 0.05                # small force so the motion stays near-linear
T_settle = 60.0                               # fast mode: short fixed settle (start is already near steady state)

# Same springs and geometry as above, but with damping, so resonance peaks have finite height.
springs_d = spring_parameters(k=springs.k, b=np.full(springs.n, b_sweep), ends=springs.ends)
springs_d.attach(container, r0)

# Small-oscillation prediction: steady state of  M xi'' + C xi' + K xi = F  in x (see equations above),
# with springs ordered [coupling, left, top0, bottom0, right, top1, bottom1].
kc, kL, kR = springs_d.k[[0, 1, 4]]
bc, bL, bR = springs_d.b[[0, 1, 4]]
K = np.array([[kL + kc, -kc], [-kc, kR + kc]])
C = np.array([[bL + bc, -bc], [-bc, bR + bc]])

def linear_response(w):
    """Complex x-amplitudes X of both masses, with xi(t) = Re[X exp(i w t)], for F0 cos(w t) on mass 0."""
    return np.linalg.solve(K - w**2 * np.diag(m.m) + 1j * w * C, [F0_sweep, 0])


def steady_amplitude(w):
    """Drive mass 0 in x at frequency w, starting from the small-oscillation steady state.
    Return max |x0 - x0_rest| and max |y - y_rest| over a window after a short settle."""
    def rhs(t, y):
        r, v = y[:2 * m.n].reshape(m.n, 2), y[2 * m.n:].reshape(m.n, 2)
        F = springs_d.forces(r, v)
        F[0, 0] += F0_sweep * np.cos(w * t)
        return np.r_[v.ravel(), (F / m.m[:, None]).ravel()]

    X = linear_response(w)
    r_start, v_start = r0.copy(), np.zeros_like(r0)
    r_start[:, 0] += X.real                   # xi(0)  = Re X
    v_start[:, 0] = -w * X.imag               # xi'(0) = Re(i w X)
    T_window = 20.0 if w == 0 else max(20.0, 2 * 2 * np.pi / w)   # at least two drive periods
    sol = solve_ivp(rhs, (0, T_settle + T_window), np.r_[r_start.ravel(), v_start.ravel()],
                    method="DOP853", rtol=1e-6, atol=1e-8, dense_output=True)
    r = sol.sol(np.linspace(T_settle, T_settle + T_window, 1000))[:2 * m.n].T.reshape(-1, m.n, 2)
    return np.abs(r[:, 0, 0] - r0[0, 0]).max(), np.abs(r[:, :, 1] - r0[:, 1]).max()


# Grid from 0 to 3, refined around the predicted x resonances at 1 and sqrt(3).
w_sweep = np.unique(np.r_[np.linspace(0, 3, 61), np.linspace(0.95, 1.1, 16), np.linspace(1.68, 1.8, 13)])
amp, amp_y = np.array([steady_amplitude(w) for w in w_sweep]).T
w_fine = np.linspace(0, 3, 3001)
amp_lin = np.array([abs(linear_response(w)[0]) for w in w_fine])

# Checks
# 1. A pure x drive never excites y (the setup is mirror-symmetric about y = H/2), so no sqrt(2) peak.
print(f"max |y - y_rest| over the whole sweep = {amp_y.max():.1e}")
assert amp_y.max() < 1e-8
# 2. Away from the peaks the motion is small, so the full simulation should match the linear prediction
#    (loose 5% margin in fast mode).
off_peak = (np.abs(w_sweep - 1) > 0.2) & (np.abs(w_sweep - np.sqrt(3)) > 0.2)
rel_err = np.abs(amp[off_peak] / np.interp(w_sweep[off_peak], w_fine, amp_lin) - 1)
print(f"off-peak max relative difference from linear theory = {rel_err.max():.1e}")
assert rel_err.max() < 0.05
# 3. The two simulated peaks sit near the predicted normal-mode frequencies.
peaks = [w_sweep[(w_sweep > lo) & (w_sweep < hi)][np.argmax(amp[(w_sweep > lo) & (w_sweep < hi)])]
         for lo, hi in [(0.8, 1.3), (1.5, 2.0)]]
print(f"simulated peaks at w = {peaks[0]:.3f}, {peaks[1]:.3f}   (linear theory: 1, {np.sqrt(3):.3f})")
assert abs(peaks[0] - 1) < 0.08 and abs(peaks[1] - np.sqrt(3)) < 0.08

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(w_fine, amp_lin, color="gray", lw=1, label="small-oscillation theory")
ax.semilogy(w_sweep, amp, ".", ms=4, label="simulation")
for w0, name in [(1, "in-phase"), (np.sqrt(3), "out-of-phase")]:
    ax.axvline(w0, ls="--", color="k", lw=0.8)
    ax.text(w0, 0.97, f" {name}", fontsize=9, va="top", transform=ax.get_xaxis_transform())
ax.set(xlabel=r"drive frequency $\omega_d$", ylabel=r"amplitude of $x_0$",
       title=f"Mass 0 driven in x,  $F_0$ = {F0_sweep},  b = {b_sweep}", xlim=(0, 3))
ax.legend(loc="lower left")
plt.show()


Ok, now we are ready to see the dynamics when applying a force to mass 0 in both the x and y directions. We will use a driving force $F(t) = F_0\cos(\omega_d t)$ with $F_0 = 0.05$ and a small damping $b = 0.05$ on every spring (with $F_0 = 1$ the masses would hit the walls near resonance). The amplitude is defined as the maximum displacement of mass $0$ in both x and y after transients have died out.

In [ ]:
w_xy, T_xy, n_frames_xy = np.sqrt(2), 60.0, 200    # drive at the y resonance; animate t = 0..T_xy
T_long = 400.0                                     # run on to here to read off the steady amplitudes


def F_xy(t):
    """F0 cos(w t) on mass 0, in x and in y."""
    F = np.zeros((m.n, 2))
    F[0] = F0_sweep * np.cos(w_xy * t)
    return F


def rhs_xy(t, y):
    """State y = [positions, velocities, work by the drive, work by the dampers]."""
    r, v = y[:2 * m.n].reshape(m.n, 2), y[2 * m.n:4 * m.n].reshape(m.n, 2)
    F_spring = springs_d.forces(r, v)
    F_damp = F_spring - springs.forces(r, v)       # springs has the same k's and b = 0
    a = (F_spring + F_xy(t)) / m.m[:, None]
    return np.r_[v.ravel(), a.ravel(), np.sum(F_xy(t) * v), np.sum(F_damp * v)]


sol = solve_ivp(rhs_xy, (0, T_long), np.r_[r0.ravel(), np.zeros(2 * m.n), 0.0, 0.0],
                method="DOP853", rtol=1e-10, atol=1e-12, dense_output=True)
t_xy = np.linspace(0, T_xy, n_frames_xy)
y_xy = sol.sol(t_xy)
r_xy = y_xy[:2 * m.n].T.reshape(-1, m.n, 2)

# Check (work-energy theorem with damping): change in kinetic + spring energy = work by drive + work by dampers,
# and the dampers only ever remove energy.
t_chk = np.linspace(0, T_long, 4000)
y_chk = sol.sol(t_chk)
r_c, v_c = y_chk[:2 * m.n].T.reshape(-1, m.n, 2), y_chk[2 * m.n:4 * m.n].T.reshape(-1, m.n, 2)
W_drive, W_damp = y_chk[-2], y_chk[-1]
E = np.array([0.5 * np.sum(m.m[:, None] * v ** 2) + springs.energy(r) for r, v in zip(r_c, v_c)])
print(f"max |E - E(0) - W_drive - W_damp| = {np.max(np.abs(E - E[0] - W_drive - W_damp)):.1e}")
assert np.allclose(E - E[0], W_drive + W_damp, atol=1e-8)
assert np.all(np.diff(W_damp) <= 1e-12)

# Steady amplitudes of mass 0 over the last two drive periods, next to the small-oscillation prediction.
t_end = np.linspace(T_long - 2 * 2 * np.pi / w_xy, T_long, 2000)
d_end = sol.sol(t_end)[:2 * m.n].T.reshape(-1, m.n, 2) - r0
Ax_lin = abs(linear_response(w_xy)[0])
Ay_lin = F0_sweep / abs(springs_d.k[2] + springs_d.k[3] - w_xy**2 * m.m[0] + 1j * w_xy * (springs_d.b[2] + springs_d.b[3]))
print(f"mass 0 steady amplitude:  x = {np.abs(d_end[:, 0, 0]).max():.4f}  (linear {Ax_lin:.4f}),"
      f"  y = {np.abs(d_end[:, 0, 1]).max():.4f}  (linear {Ay_lin:.4f})")
print(f"mass 1 steady amplitude:  x = {np.abs(d_end[:, 1, 0]).max():.4f},  y = {np.abs(d_end[:, 1, 1]).max():.4f}")

animate(t_xy, r_xy, f"mass 0 driven in x and y,  $\\omega_d$ = {w_xy:.3f}")
